# Smart Data Downloader for Evo2_Clinical

This notebook demonstrates a smart downloader that checks if files already exist before downloading them.
The downloader will:

1. Check if the requested file already exists in the data directory
2. Only download files that are missing
3. Decompress .gz files if needed
4. Track download status and log results

This approach saves bandwidth and time by avoiding redundant downloads.

In [2]:
import os
import sys
import logging
import gzip
import shutil
import urllib.request
from pathlib import Path
from tqdm import tqdm

class DownloadProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

class SmartDownloader:
    def __init__(self, data_dir='data'):
        self.data_dir = data_dir
        self.setup_logging()
        self.setup_directories()
        
    def setup_logging(self):
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            handlers=[
                logging.StreamHandler(),
                logging.FileHandler('database_download.log')
            ]
        )
        self.logger = logging.getLogger("SmartDownloader")

    def setup_directories(self):
        """Create necessary directories if they don't exist."""
        dirs = [
            f'{self.data_dir}/reference',
            f'{self.data_dir}/1000genomes',
            f'{self.data_dir}/encode',
            f'{self.data_dir}/gwas'
        ]
        for dir_path in dirs:
            Path(dir_path).mkdir(parents=True, exist_ok=True)

    def download_file(self, url, output_path, force=False):
        """Smart download function that checks if file exists before downloading.
        
        Args:
            url: URL to download from
            output_path: Local path to save the file
            force: If True, download even if file exists
            
        Returns:
            Path to the downloaded file or None if download failed
        """
        # Check if file already exists
        if os.path.exists(output_path) and not force:
            self.logger.info(f"File already exists: {output_path}")
            return output_path
        
        # If not, download it
        self.logger.info(f"Downloading {url} to {output_path}...")
        try:
            with DownloadProgressBar(unit='B', unit_scale=True,
                                   miniters=1, desc="Downloading") as t:
                urllib.request.urlretrieve(url, filename=output_path,
                                         reporthook=t.update_to)
            self.logger.info(f"Successfully downloaded: {output_path}")
            return output_path
        except Exception as e:
            self.logger.error(f"Error downloading {url}: {str(e)}")
            return None

    def decompress_file(self, gz_path, force=False):
        """Decompress a gzipped file if the decompressed version doesn't exist."""
        output_path = str(gz_path)[:-3]  # Remove .gz extension
        
        # Check if decompressed file already exists
        if os.path.exists(output_path) and not force:
            self.logger.info(f"Decompressed file already exists: {output_path}")
            return output_path
        
        try:
            with gzip.open(gz_path, 'rb') as f_in:
                with open(output_path, 'wb') as f_out:
                    shutil.copyfileobj(f_in, f_out)
            self.logger.info(f"Decompressed {gz_path} to {output_path}")
            return output_path
        except Exception as e:
            self.logger.error(f"Error decompressing {gz_path}: {str(e)}")
            return None

    def download_and_process(self, url, output_path, decompress=True, force=False):
        """Download and optionally decompress a file.
        
        Args:
            url: URL to download from
            output_path: Local path to save the file
            decompress: Whether to decompress the file if it's gzipped
            force: If True, download even if file exists
            
        Returns:
            Path to the final file (decompressed if applicable)
        """
        # Check if output path or decompressed file already exists
        decompressed_path = output_path[:-3] if output_path.endswith('.gz') else None
        
        if decompress and decompressed_path and os.path.exists(decompressed_path) and not force:
            self.logger.info(f"Final file already exists: {decompressed_path}")
            return decompressed_path
        
        # Download the file if needed
        downloaded_path = self.download_file(url, output_path, force)
        if not downloaded_path:
            return None
        
        # Decompress if needed
        if decompress and output_path.endswith('.gz'):
            return self.decompress_file(downloaded_path, force)
        
        return downloaded_path

    def download_dataset(self, dataset_name, force=False):
        """Download a specific dataset by name.
        
        Args:
            dataset_name: Name of the dataset to download
                (one of: 'gene_annotations', 'reference_genome', 
                '1000g_data', 'encode_data', 'gwas_catalog')
            force: If True, force download even if files exist
            
        Returns:
            List of paths to downloaded/processed files
        """
        if dataset_name == "gene_annotations":
            url = "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/genes/hg38.ncbiRefSeq.gtf.gz"
            output_path = f"{self.data_dir}/reference/GRCh38.genes.gtf.gz"
            return [self.download_and_process(url, output_path, force=force)]
            
        elif dataset_name == "reference_genome":
            url = "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr19.fa.gz"
            output_path = f"{self.data_dir}/reference/GRCh38.chr19.fa.gz"
            return [self.download_and_process(url, output_path, force=force)]
            
        elif dataset_name == "1000g_data":
            url = "https://ftp.ncbi.nlm.nih.gov/1000genomes/ftp/release/20130502/ALL.chr19.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz"
            output_path = f"{self.data_dir}/1000genomes/chr19_variants.vcf.gz"
            return [self.download_and_process(url, output_path, force=force)]
            
        elif dataset_name == "encode_data":
            urls = [
                "https://www.encodeproject.org/files/ENCFF017XLW/@@download/ENCFF017XLW.bed.gz",  # HUVEC H3K27ac ChIP-seq peaks
                "https://www.encodeproject.org/files/ENCFF721JMB/@@download/ENCFF721JMB.bed.gz"   # HUVEC DNase-seq peaks
            ]
            results = []
            for i, url in enumerate(urls, 1):
                output_path = f"{self.data_dir}/encode/endothelial_peaks_{i}.bed.gz"
                result = self.download_and_process(url, output_path, force=force)
                if result:
                    results.append(result)
            return results
            
        elif dataset_name == "gwas_catalog":
            url = "https://www.ebi.ac.uk/gwas/api/search/downloads/full"
            output_path = f"{self.data_dir}/gwas/gwas-catalog-associations.tsv"
            return [self.download_and_process(url, output_path, decompress=False, force=force)]
            
        else:
            self.logger.error(f"Unknown dataset name: {dataset_name}")
            return []

    def download_all(self, force=False):
        """Download all datasets."""
        datasets = [
            "gene_annotations",
            "reference_genome",
            "1000g_data",
            "encode_data",
            "gwas_catalog"
        ]
        
        all_results = {}
        for dataset in datasets:
            self.logger.info(f"Processing dataset: {dataset}")
            results = self.download_dataset(dataset, force=force)
            all_results[dataset] = results
            
        return all_results

In [3]:
# Example usage of the SmartDownloader

# Initialize the downloader
downloader = SmartDownloader()

# Download a specific dataset (if not already present)
print("Downloading reference genome (chr19)...")
results = downloader.download_dataset("reference_genome")
print(f"Downloaded/processed files: {results}\n")

# Download ENCODE data (if not already present)
print("Downloading ENCODE data...")
results = downloader.download_dataset("encode_data")
print(f"Downloaded/processed files: {results}\n")

# You can also force download even if files exist
# For example: downloader.download_dataset("gwas_catalog", force=True)

# Check which datasets are available
available_datasets = [
    "gene_annotations", 
    "reference_genome", 
    "1000g_data", 
    "encode_data", 
    "gwas_catalog"
]
print("Available datasets:")
for dataset in available_datasets:
    print(f"- {dataset}")

2025-04-01 20:18:06,705 - SmartDownloader - INFO - Final file already exists: data/reference/GRCh38.chr19.fa
2025-04-01 20:18:06,706 - SmartDownloader - INFO - Final file already exists: data/encode/endothelial_peaks_1.bed
2025-04-01 20:18:06,709 - SmartDownloader - INFO - Final file already exists: data/encode/endothelial_peaks_2.bed


Downloaded/processed files: ['data/reference/GRCh38.chr19.fa']

Downloaded/processed files: ['data/encode/endothelial_peaks_1.bed', 'data/encode/endothelial_peaks_2.bed']

Available datasets:
- gene_annotations
- reference_genome
- 1000g_data
- encode_data
- gwas_catalog


## Downloading Dataset on Demand in Data Analysis Workflow

This example shows how to integrate the smart downloader into your analysis pipeline, ensuring that data is downloaded only when needed.

In [4]:
def get_vcf_path(chromosome="19", download_if_missing=True):
    """Get the path to a VCF file, downloading it if necessary.
    
    Args:
        chromosome: Chromosome number (default is "19")
        download_if_missing: Whether to download the file if it's missing
        
    Returns:
        Path to the VCF file
    """
    vcf_path = f"data/1000genomes/chr{chromosome}_variants.vcf"
    
    # Check if file exists
    if not os.path.exists(vcf_path):
        if download_if_missing:
            print(f"VCF file for chromosome {chromosome} not found. Downloading...")
            downloader = SmartDownloader()
            downloader.download_dataset("1000g_data")
        else:
            raise FileNotFoundError(f"VCF file not found: {vcf_path}")
    
    return vcf_path

# Example usage in your analysis pipeline
def run_variant_analysis(chromosome="19"):
    """Run variant analysis for a specific chromosome."""
    # Get VCF path, downloading if necessary
    try:
        vcf_path = get_vcf_path(chromosome)
        
        # Now use the VCF file in your analysis
        print(f"Running analysis on {vcf_path}")
        
        # Example: load the VCF file using the variant_analysis module
        processor = va.VariantProcessor()
        variants_df = processor.load_vcf(vcf_path)
        
        print(f"Loaded {len(variants_df)} variants from chromosome {chromosome}")
        return variants_df
        
    except Exception as e:
        print(f"Error in variant analysis: {str(e)}")
        return None

# Run the analysis
variants_df = run_variant_analysis()

Running analysis on data/1000genomes/chr19_variants.vcf
Error in variant analysis: name 'va' is not defined


In [5]:
# Usage examples for running this app.

In [6]:
# #!/usr/bin/env python3
# """
# Variant Analysis Module for Evo2 Pipeline

# This module provides utilities for analyzing genomic variants, including:
# - Processing VCF files from 1000 Genomes and custom sources
# - Analyzing variants in any gene or genomic region
# - Scoring variant effects using various computational methods
# - Integrating ENCODE data for cell-type specific analysis

# Author: Shabab Khan
# Date: March 26, 2025
# """

# import pandas as pd
# import numpy as np
# from typing import Dict, List, Tuple, Optional, Union
# import logging
# from pathlib import Path
# import cyvcf2
# import allel
# import pysam
# from Bio import SeqIO
# from pybedtools import BedTool

# # Configure logging
# logging.basicConfig(
#     level=logging.INFO,
#     format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
# )
# logger = logging.getLogger("VariantAnalysis")

# class VariantProcessor:
#     """Class for processing and analyzing genomic variants."""
    
#     def __init__(self, reference_genome: str = "GRCh38"):
#         self.reference_genome = reference_genome
#         self.gene_annotations = {}  # Will be populated with gene coordinates
#         logger.info(f"Initialized VariantProcessor with {reference_genome}")

#     def load_vcf(self, vcf_path: str) -> pd.DataFrame:
#         """Load and parse VCF file into a DataFrame."""
#         vcf = cyvcf2.VCF(vcf_path)
#         variants = []
        
#         for variant in vcf:
#             variants.append({
#                 'chrom': variant.CHROM,
#                 'pos': variant.POS,
#                 'id': variant.ID,
#                 'ref': variant.REF,
#                 'alt': ','.join(variant.ALT),
#                 'qual': variant.QUAL,
#                 'filter': variant.FILTER,
#                 # Placeholder for gene annotation
#                 'gene': self.annotate_gene(variant.CHROM, variant.POS)
#             })
            
#         return pd.DataFrame(variants)

#     def annotate_gene(self, chrom: str, pos: int) -> str:
#         """Annotate a variant with gene information based on its position.
        
#         Args:
#             chrom: Chromosome of the variant
#             pos: Position of the variant
            
#         Returns:
#             Gene name as string
#         """
#         for gene, coords in self.gene_annotations.items():
#             if (chrom == coords['chrom'] and 
#                 coords['start'] <= pos <= coords['end']):
#                 return gene
#         return "Unknown"

#     def annotate_variants(self, variants_df: pd.DataFrame, 
#                          encode_data: Optional[pd.DataFrame] = None) -> pd.DataFrame:
#         """Annotate variants with functional information and ENCODE data.
        
#         Args:
#             variants_df: DataFrame of variants
#             encode_data: Optional DataFrame with ENCODE annotations
            
#         Returns:
#             Annotated variant DataFrame
#         """
#         # Add basic annotations
#         annotated_df = variants_df.copy()
        
#         # Add ENCODE data if available
#         if encode_data is not None:
#             annotated_df = pd.merge(
#                 annotated_df, 
#                 encode_data,
#                 how='left',
#                 on=['chrom', 'pos']
#             )
            
#         return annotated_df

#     def score_variant_impact(self, variant_info: Dict) -> float:
#         """Score the potential functional impact of a variant.
        
#         Args:
#             variant_info: Dictionary containing variant information
            
#         Returns:
#             Impact score between 0 and 1
#         """
#         impact_score = 0.0

#         # Score based on variant type
#         variant_type = variant_info.get("variant_type", "").upper()
#         if variant_type == "SNP":
#             impact_score += 0.2
#         elif variant_type == "INDEL":
#             impact_score += 0.3
#         elif variant_type == "SV":
#             impact_score += 0.4

#         # Add scores based on genomic context
#         if variant_info.get("in_exon"):
#             impact_score += 0.3
#         elif variant_info.get("in_promoter"):
#             impact_score += 0.2
#         elif variant_info.get("in_enhancer"):
#             impact_score += 0.15

#         # Normalize score to be between 0 and 1
#         return min(impact_score, 1.0)

#     def analyze_gene_variants(self, variants_df: pd.DataFrame,
#                             gene_id: str,
#                             region_info: Optional[Dict] = None) -> pd.DataFrame:
#         """Analyze variants in a specific gene or genomic region.
        
#         Args:
#             variants_df: DataFrame of variants
#             gene_id: ID of the gene to analyze
#             region_info: Optional dictionary with region coordinates
            
#         Returns:
#             DataFrame with variant analysis
#         """
#         # Filter for gene variants
#         if region_info:
#             gene_variants = variants_df[
#                 (variants_df['chrom'] == region_info['chrom']) &
#                 (variants_df['pos'] >= region_info['start']) &
#                 (variants_df['pos'] <= region_info['end'])
#             ].copy()
#         else:
#             gene_variants = variants_df[
#                 variants_df['gene'] == gene_id
#             ].copy()

#         # Add analysis columns
#         gene_variants["analyzed_gene"] = gene_id
#         gene_variants["impact_score"] = gene_variants.apply(
#             lambda row: self.score_variant_impact(row.to_dict()),
#             axis=1
#         )

#         return gene_variants

#     def analyze_gwas_variants(self, variants_df: pd.DataFrame,
#                             gwas_catalog: pd.DataFrame) -> pd.DataFrame:
#         """Analyze variants based on GWAS catalog data.
        
#         Args:
#             variants_df: DataFrame of variants
#             gwas_catalog: DataFrame with GWAS annotations
            
#         Returns:
#             DataFrame with GWAS variant analysis
#         """
#         # Merge variants with GWAS data
#         gwas_variants = pd.merge(
#             variants_df,
#             gwas_catalog,
#             how='left',
#             left_on='variant_id',
#             right_on='SNPS'
#         )
        
#         return gwas_variants


# def main():
#     """Main function to demonstrate variant analysis functionality."""
#     processor = VariantProcessor()
    
#     # Example usage
#     variants_df = processor.load_vcf("example.vcf")
#     annotated_df = processor.annotate_variants(variants_df)
    
#     # Generate report
#     processor.generate_variant_report(annotated_df, "variant_report.txt")


# if __name__ == "__main__":
#     main()

In [7]:
import variant_analysis as va
import visualization as vis

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
variants_df = va.VariantProcessor().load_vcf("data/1000genomes/chr19_variants.vcf")
# Analysis of variants in a specific gene
gene_name = "BRCA1"
gene_variants = variants_df[variants_df['gene'] == gene_name]
print(f"Found {len(gene_variants)} variants in gene {gene_name}")


TypeError: 'NoneType' object is not subscriptable

In [10]:

# Example of scoring variant impact
impact_scores = gene_variants.apply(lambda row: analysis.score_variant_impact(row.to_dict()), axis=1)
gene_variants["impact_score"] = impact_scores
print(f"Impact scores calculated for {len(gene_variants)} variants")

analysis = va.VariantProcessor()

# Create a visualization object
variants_df = analysis.load_vcf("data/1000genomes/chr19_variants.vcf")
print(f"Loaded {len(variants_df)} variants")


plt.figure(figsize=(10, 6))
plt.hist(gene_variants['variant_quality'], bins=30, alpha=0.7, color='blue')
plt.title(f'Quality Distribution of Variants in {gene_name}')
plt.xlabel('Variant Quality')
plt.ylabel('Frequency')
plt.grid(True)
plt.tight_layout()
plt.show()

NameError: name 'gene_variants' is not defined

In [ ]:
import visualization as vis
visualizer = vis.VariantVisualizer()

# Plot distributions
visualizer.plot_variant_distribution(variants_df)
plt.show()

# Plot impact scores (note: impact scores might need to be calculated first)
annotated_df = analysis.annotate_variants(variants_df)
# Optionally add impact scores if they're not automatically added in annotate_variants
annotated_df["impact_score"] = annotated_df.apply(lambda row: analysis.score_variant_impact(row.to_dict()), axis=1)
visualizer.plot_impact_scores(annotated_df)
plt.show()
visualizer.plot_impact_scores(annotated_df)
plt.show()

# Plot variant heatmap
visualizer.plot_variant_heatmap(variants_df)

OSError: 'seaborn' is not a valid package style, path of style file, URL of style file, or library style name (library styles are listed in `style.available`)